# Prediksi foto baru: LBF + multi-radius, model multi_fixed
Notebook ini memakai model **`results_lbf_multiradius/models/multi_fixed.joblib`**,
bukan model MediaPipe atau konfigurasi tuned. Tidak ada training atau tuning ulang.
Model dimuat dari file hasil training terbaru pada path tersebut setiap Run All.
Mask mengecualikan hidung (titik 27-35), sama seperti training terbaru.

**Cara pakai:** pilih kernel **Python (ACNE04 Landmark)**, ubah nama foto di
langkah 2, lalu Run All. Notebook mandiri; notebook training tidak perlu dijalankan.

Alur: **foto → crop Haar → mask LBF → LBP radius 1,2,3 + HSV → 102 fitur →
pipeline scaler + SVM → prediksi kelas**.

Letakkan foto asli pada folder `haar_lbf_lbp_hsv_svm`. Jangan memakai foto
yang sudah diberi overlay hijau atau padding. Untuk mencoba foto lain, ubah langkah
2 kemudian jalankan kembali langkah 2 sampai 6. Keluaran adalah prediksi model;
satu foto tanpa label acuan tidak cukup untuk mengukur akurasi.

## 1. Import library dan muat kedua model
Ada dua model berbeda:

- **`lbfmodel.yaml`** mencari 68 titik wajah, bukan kelas jerawat.
- **`multi_fixed.joblib`** memuat StandardScaler dan SVM untuk klasifikasi jerawat.

Konfigurasi training digunakan untuk menentukan radius dan preprocessing.
Sidik jari SHA-256 memastikan file landmark dan Haar sama dengan saat training.
Tidak diperlukan Dlib, MediaPipe atau GPU.

In [ ]:
from pathlib import Path
import json
import hashlib

import cv2
import numpy as np
import matplotlib.pyplot as plt
import joblib
from PIL import Image, ImageOps
from skimage.feature import local_binary_pattern

# Temukan root proyek dari lokasi notebook saat ini.
ROOT = Path.cwd()
while not (ROOT / 'results_lbf_multiradius/models/multi_fixed.joblib').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Model multi_fixed LBF tidak ditemukan. Buka notebook dari dalam folder haar_lbf_lbp_hsv_svm.')
    ROOT = ROOT.parent

PROJECT = ROOT
RESULTS = PROJECT / 'results_lbf_multiradius'
MODEL_PATH = RESULTS / 'models/multi_fixed.joblib'
CONFIG_PATH = RESULTS / 'models/multi_fixed_config.json'
settings = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
metadata = json.loads((RESULTS / 'run_metadata.json').read_text(encoding='utf-8'))
model = joblib.load(MODEL_PATH)

assert hasattr(cv2, 'face') and hasattr(cv2.face, 'createFacemarkLBF'), \
    'Pilih kernel Python (ACNE04 Landmark) dengan OpenCV contrib.'
assert settings['landmark_method'] == 'opencv_facemark_lbf_68'
assert model.n_features_in_ == settings['feature_count'] == 102
assert settings['lbp_radii'] == [1, 2, 3]
assert settings['lbp_neighbors'] == '8 * radius'
assert settings['lbp_method'] == 'uniform'
assert settings['lbp_min_interior_pixels'] == 100
assert settings['hsv_bins_per_channel'] == 16
assert model.named_steps['svc'].C == settings['C']
assert model.named_steps['svc'].gamma == settings['gamma']

CONFIG = settings['landmark_options']
RADII = settings['lbp_radii']
DRAFT_SIZE = settings['jpeg_draft_size']
DETECTION_SIZE = settings['haar_detection_max_side']
CROP_SIZE = settings['crop_max_side']
FACE_MARGIN = settings['haar_margin']
LABELS = {0: 'Ringan (mild)', 1: 'Sedang (moderate)',
          2: 'Berat (severe)', 3: 'Sangat berat (very severe)'}

# Fungsi landmark dari training memakai parameter Haar berikut.
assert settings['haar_scale_factor'] == 1.1
assert settings['haar_min_neighbors'] == 5
assert settings['haar_min_size'] == [40, 40]

cascade_path = Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'
assert hashlib.sha256(cascade_path.read_bytes()).hexdigest() == metadata['cascade_sha256']
detector = cv2.CascadeClassifier(str(cascade_path))
assert not detector.empty(), 'File Haar gagal dimuat.'

LBF_PATH = PROJECT / 'lbf_models' / settings['landmark_predictor_file']
if not LBF_PATH.is_file():
    raise FileNotFoundError('lbfmodel.yaml belum ada. Jalankan download_lbf_model.py.')
assert hashlib.sha256(LBF_PATH.read_bytes()).hexdigest() == settings['landmark_predictor_sha256'], \
    'File LBF berbeda dari training. Gunakan file landmark yang sama.'
facemark = cv2.face.createFacemarkLBF()
facemark.loadModel(str(LBF_PATH))
cv2.setNumThreads(1)

print('Model klasifikasi:', MODEL_PATH.resolve())
print('Mask: mata, alis, hidung (27-35), dan bibir dikeluarkan.')
print('Model landmark:', LBF_PATH.name)
print('Radius:', RADII, '| fitur:', model.n_features_in_)
print('C:', settings['C'], '| gamma:', settings['gamma'])

## 2. Pilih foto
Ubah nama file di bawah sesuai foto pada folder `haar_lbf_lbp_hsv_svm`.
Untuk lokasi lain, gunakan path lengkap, contohnya
`IMAGE_PATH = Path(r"C:\Users\damar\Pictures\wajah.jpg")`.
`Path(PROJECT)` menunjuk folder notebook ini, bukan root proyek.

In [ ]:
# UBAH nama file berikut untuk mencoba foto lain.
IMAGE_PATH = Path(PROJECT) / 'gun_kanan_mulus.png'

if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f'Foto tidak ditemukan. Periksa IMAGE_PATH: {IMAGE_PATH}')
print('Foto yang diproses:', IMAGE_PATH.name)

## 3. Baca foto dan crop wajah
Sama seperti training: JPEG draft 768 → koreksi EXIF → RGB → Haar pada grayscale
maksimal 640 → kotak terbesar → margin 15% → crop RGB → resize proporsional
maksimal 384 tanpa padding. Nilainya dibaca dari konfigurasi.

Jika Haar gagal, seluruh foto di-resize tanpa padding sebagai fallback. Ini setara
dengan isi foto baseline training, tanpa membutuhkan manifest untuk foto baru.

In [ ]:
def read_and_crop(image_path):
    with Image.open(image_path) as source:
        source.draft('RGB', (DRAFT_SIZE, DRAFT_SIZE))
        original = ImageOps.exif_transpose(source).convert('RGB')

    small = ImageOps.contain(original, (DETECTION_SIZE, DETECTION_SIZE))
    gray = cv2.cvtColor(np.array(small), cv2.COLOR_RGB2GRAY)
    boxes = detector.detectMultiScale(
        gray,
        scaleFactor=settings['haar_scale_factor'],
        minNeighbors=settings['haar_min_neighbors'],
        minSize=tuple(settings['haar_min_size']),
    )

    if len(boxes) == 0:
        crop = ImageOps.contain(original, (CROP_SIZE, CROP_SIZE), Image.Resampling.LANCZOS)
        return np.array(crop), False

    largest_box = None
    largest_area = 0
    for box in boxes:
        area = int(box[2]) * int(box[3])
        if area > largest_area:
            largest_area = area
            largest_box = box
    x, y, width, height = largest_box

    # Ubah koordinat kotak dari gambar kecil ke foto RGB, dengan margin yang sama.
    left = max(0, int((x - FACE_MARGIN * width) * original.width / small.width))
    top = max(0, int((y - FACE_MARGIN * height) * original.height / small.height))
    right = min(original.width, int(np.ceil((x + (1 + FACE_MARGIN) * width) * original.width / small.width)))
    bottom = min(original.height, int(np.ceil((y + (1 + FACE_MARGIN) * height) * original.height / small.height)))
    crop = original.crop((left, top, right, bottom))
    crop = ImageOps.contain(crop, (CROP_SIZE, CROP_SIZE), Image.Resampling.LANCZOS)
    return np.array(crop), True

In [ ]:
crop, haar_detected = read_and_crop(IMAGE_PATH)
print('Haar mendeteksi wajah:', haar_detected)
print('Ukuran crop:', crop.shape[1], 'x', crop.shape[0])

## 4. Buat mask LBF yang sama dengan training
Fungsi berikut disalin dari notebook training LBF:

1. Haar mencari kotak wajah di dalam crop untuk diberikan kepada LBF.
2. `facemark.fit()` mencocokkan 68 titik pada foto, bukan melatih ulang model.
3. Rahang dan perkiraan dahi membentuk mask. Batas dahi diperkirakan dari alis
   yang digeser ke atas sebesar 25% jarak vertikal rata-rata alis ke dagu.
4. Mata, alis, hidung (titik 27-35), dan bibir dikeluarkan dengan convex hull.
5. Landmark gagal atau interior mask terlalu kecil → seluruh crop sebagai fallback.

Putih berarti piksel dipilih, hitam berarti dikeluarkan. Mask ini perkiraan
geometris; rambut dapat ikut masuk dan dahi bisa terpotong. Status fallback
ditampilkan agar overlay hijau tidak disalahartikan sebagai kulit yang pasti.

In [ ]:
def effective_mask(image, mask):
    assert mask.dtype == np.uint8
    assert mask.shape == image.shape[:2]
    kernel = np.ones((3, 3), dtype=np.uint8)
    interior = cv2.erode(mask, kernel, borderType=cv2.BORDER_CONSTANT, borderValue=0)
    fallback = np.count_nonzero(interior) < 100

    if fallback:
        mask = np.full(image.shape[:2], 255, dtype=np.uint8)
    return mask, fallback

In [ ]:
def box_area(box):
    return int(box[2]) * int(box[3])

In [ ]:
def mask_from_points(image_shape, points):
    height, width = image_shape[:2]
    jaw = points[0:17]

    # LBF 68 tidak punya titik kontur dahi: buat batas atas perkiraan dari alis.
    eyebrows = points[17:27]
    eyebrow_y = float(np.mean(eyebrows[:, 1]))
    chin_y = float(points[8, 1])
    face_height = max(1.0, chin_y - eyebrow_y)
    forehead = eyebrows[::-1].copy()
    upward_shift = int(round(CONFIG['forehead_ratio'] * face_height))
    forehead[:, 1] = forehead[:, 1] - upward_shift

    contour = np.vstack([jaw, forehead]).astype(np.int32)
    contour[:, 0] = np.clip(contour[:, 0], 0, width - 1)
    contour[:, 1] = np.clip(contour[:, 1], 0, height - 1)
    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.fillPoly(mask, [contour], 255)

    excluded_regions = [points[36:42], points[42:48],
                        points[17:22], points[22:27], points[27:36], points[48:60]]
    for region in excluded_regions:
        hull = cv2.convexHull(region)
        cv2.fillConvexPoly(mask, hull, 0)
    return mask, contour

In [ ]:
def landmark_mask(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    boxes = detector.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    info = {
        'landmark_face_count': len(boxes),
        'landmark_detected': False,
        'landmark_fallback': True,
        'landmark_raw_fraction': 0.0,
        'landmark_reason': 'no_face',
    }
    full_mask = np.full(image.shape[:2], 255, dtype=np.uint8)
    if len(boxes) == 0:
        return full_mask, info, None

    selected_boxes = sorted(boxes, key=box_area, reverse=True)
    selected_boxes = selected_boxes[:CONFIG['max_num_faces']]
    rectangles = np.array(selected_boxes, dtype=np.int32).reshape(-1, 4)
    success, landmarks = facemark.fit(gray, rectangles)
    if not success or landmarks is None:
        info['landmark_reason'] = 'fit_failed'
        return full_mask, info, None

    height, width = image.shape[:2]
    selected_points = None
    selected_mask = None
    largest_area = 0
    for face_points in landmarks:
        coordinates = np.asarray(face_points, dtype=float)
        if coordinates.size != 136 or not np.isfinite(coordinates).all():
            continue
        points = np.rint(coordinates.reshape(68, 2)).astype(np.int32)
        points[:, 0] = np.clip(points[:, 0], 0, width - 1)
        points[:, 1] = np.clip(points[:, 1], 0, height - 1)
        raw_mask, contour = mask_from_points(image.shape, points)
        area = abs(cv2.contourArea(contour))
        if area > largest_area:
            largest_area = area
            selected_points = points
            selected_mask = raw_mask

    if selected_points is None:
        info['landmark_reason'] = 'invalid_geometry'
        return full_mask, info, None

    mask, fallback = effective_mask(image, selected_mask)
    info['landmark_detected'] = True
    info['landmark_fallback'] = bool(fallback)
    info['landmark_raw_fraction'] = float(np.mean(selected_mask > 0))
    if fallback:
        info['landmark_reason'] = 'small_mask'
    else:
        info['landmark_reason'] = 'ok'
    return mask, info, selected_points


def overlay_mask(image, mask):
    overlay = image.copy()
    selected = mask > 0
    green = np.array([0, 255, 0])
    overlay[selected] = (0.65 * image[selected] + 0.35 * green).astype(np.uint8)
    return overlay

In [ ]:
mask, mask_info, points = landmark_mask(crop)
print('Landmark LBF terdeteksi:', mask_info['landmark_detected'])
print('Status mask:', mask_info['landmark_reason'])
print('Fallback mask:', mask_info['landmark_fallback'])

## 5. Hitung 102 fitur lalu prediksi
Urutan fitur sama dengan training: **10 LBP radius 1 → 18 LBP radius 2 →
26 LBP radius 3 → 16 H → 16 S → 16 V**, total 102.

LBP dihitung pada grayscale asli. Mask dierosi 3×3, 5×5, 7×7 sesuai radius.
Jika interior suatu radius kurang dari 100 piksel, hanya LBP radius itu memakai
interior seluruh crop. HSV tetap memakai mask efektif awal. Setiap histogram
dinormalisasi sendiri.

Fitur satu foto berbentuk `(102,)`, lalu diubah menjadi `(1, 102)` untuk model.
Jangan melakukan `fit()` atau scaling tambahan; pipeline model sudah menanganinya.

In [ ]:
def extract_features(image, mask, radii):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    feature_parts = []
    radius_fallbacks = {}

    for radius in radii:
        neighbor_count = 8 * radius
        kernel_size = 2 * radius + 1
        kernel = np.ones((kernel_size, kernel_size), dtype=np.uint8)
        interior = cv2.erode(mask, kernel, borderType=cv2.BORDER_CONSTANT, borderValue=0)

        fallback = np.count_nonzero(interior) < 100
        if fallback:
            full = np.full(image.shape[:2], 255, dtype=np.uint8)
            interior = cv2.erode(full, kernel, borderType=cv2.BORDER_CONSTANT, borderValue=0)
        radius_fallbacks[radius] = bool(fallback)
        if np.count_nonzero(interior) == 0:
            raise ValueError('Crop terlalu kecil untuk radius LBP yang dipilih.')

        lbp = local_binary_pattern(gray, neighbor_count, radius, method='uniform')
        selected_values = lbp[interior > 0].astype(int)
        histogram = np.bincount(selected_values, minlength=neighbor_count + 2).astype(float)
        histogram = histogram / histogram.sum()
        feature_parts.append(histogram)

    # Histogram HSV memakai mask efektif yang sama untuk kedua varian.
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    upper_limits = [180, 256, 256]
    for channel in range(3):
        histogram = cv2.calcHist([hsv], [channel], mask, [16], [0, upper_limits[channel]])
        histogram = histogram.ravel().astype(float)
        histogram = histogram / histogram.sum()
        feature_parts.append(histogram)

    vector = np.concatenate(feature_parts)
    assert np.isfinite(vector).all()
    return vector, radius_fallbacks

In [ ]:
features, radius_fallbacks = extract_features(crop, mask, RADII)
assert features.shape == (settings['feature_count'],)
model_input = features.reshape(1, -1)
prediction = int(model.predict(model_input)[0])

print('Bentuk input model:', model_input.shape)
print('Prediksi:', LABELS[prediction])
print('Fallback LBP per radius:', radius_fallbacks)

if not haar_detected:
    print('Catatan: Haar gagal; seluruh foto yang sudah di-resize menjadi crop fallback.')
if mask_info['landmark_fallback']:
    print('Catatan: mask LBF fallback; seluruh crop dipilih.')
for radius in RADII:
    if radius_fallbacks[radius]:
        print('Catatan: LBP radius', radius, 'memakai interior seluruh crop.')

## 6. Lihat hasil dan area yang dianalisis
Hijau hanya overlay penjelas; fitur menggunakan warna asli foto. Mask efektif
menunjukkan area HSV dan mask awal LBP. Jika ada fallback LBP per radius, area
tekstur radius itu mengikuti catatan di atas.

Model mengeluarkan label kelas, bukan probabilitas. Periksa crop, dahi, rambut,
dan status fallback sebelum menafsirkan prediksi.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(crop)
axes[0].set_title('Crop Haar / fallback')
axes[1].imshow(mask, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Mask efektif LBF')
axes[2].imshow(overlay_mask(crop, mask))
axes[2].set_title('Area HSV / mask awal LBP')
for axis in axes:
    axis.axis('off')
figure.suptitle('Prediksi: ' + LABELS[prediction])
plt.tight_layout()
plt.show()